# 🎬 Cloud Video Generator (Wan 2.1 Optimized)
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
### High-Performance T2V & I2V for T4 GPUs
Implements the **Alibaba Wan-AI** video generation pipeline using `device_map='auto'` to distribute load across **GPU VRAM + System RAM + CPU** — fully compatible with **Colab T4** and **Kaggle T4**.

## 🛠️ Step 1: Environment Setup & Dependency Installation

In [ ]:
import os, sys
# ── Environment Detection (Kaggle-First) ────────────────────────────────────────
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    ENV = "Kaggle"
else:
    try:
        import google.colab
        ENV = "Colab"
    except ImportError:
        ENV = "Local"

print(f"✅ Detected: {ENV}")
# Kaggle requires Internet to be enabled in Settings → Internet
if ENV == "Kaggle":
    print("ℹ️  Kaggle: Make sure 'Internet' is ON in the right-side panel.")

# ── Install Latest Dependencies ────────────────────────────────────────────────
!pip install -q -U diffusers transformers accelerate xformers moviepy sentencepiece gradio
print("✅ Dependencies installed.")

## 🧠 Step 2: Model Configuration & Loading
Uses `device_map='auto'` (via `accelerate`) to intelligently spread model layers across **VRAM → RAM → CPU**, eliminating OOM crashes.

In [ ]:
import torch, gc, os, uuid
from diffusers import WanPipeline, WanImageToVideoPipeline
from diffusers.utils import export_to_video
import gradio as gr
from PIL import Image

# ── Config ─────────────────────────────────────────────────────────────────────
T2V_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
I2V_MODEL_ID = "Wan-AI/Wan2.1-I2V-1.3B-Diffusers"
t2v_pipe = None
i2v_pipe = None

def print_memory():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  🖥️  VRAM: {used:.2f} GB / {total:.2f} GB")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def load_pipeline(mode="t2v"):
    global t2v_pipe, i2v_pipe
    clear_memory()
    if mode == "t2v":
        if t2v_pipe is None:
            print(f"⏳ Loading T2V: {T2V_MODEL_ID}")
            t2v_pipe = WanPipeline.from_pretrained(
                T2V_MODEL_ID,
                torch_dtype=torch.float16,
                use_safetensors=True,
                device_map="auto"
            )
            t2v_pipe.enable_attention_slicing()
            try:
                t2v_pipe.enable_xformers_memory_efficient_attention()
                print("  ✅ Xformers enabled.")
            except Exception:
                print("  ⚠️  Xformers unavailable, using default attention.")
            print("  ✅ T2V model ready."); print_memory()
        return t2v_pipe
    else:
        if i2v_pipe is None:
            print(f"⏳ Loading I2V: {I2V_MODEL_ID}")
            i2v_pipe = WanImageToVideoPipeline.from_pretrained(
                I2V_MODEL_ID,
                torch_dtype=torch.float16,
                use_safetensors=True,
                device_map="auto"
            )
            i2v_pipe.enable_attention_slicing()
            print("  ✅ I2V model ready."); print_memory()
        return i2v_pipe

print("✅ Loader functions ready.")

## 🎨 Step 3: Interactive Gradio Web UI
Launch the dashboard to generate videos from text prompts or animate images.

In [ ]:
def generate_t2v(prompt, negative_prompt, steps, guidance, num_frames, seed, progress=gr.Progress(track_tqdm=True)):
    if not prompt.strip():
        return None, "❌ Please enter a prompt."
    try:
        pipe = load_pipeline("t2v")
        gen = torch.Generator("cuda" if torch.cuda.is_available() else "cpu")
        if seed != -1: gen.manual_seed(int(seed))
        output = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            num_frames=int(num_frames),
            height=480, width=832,
            num_inference_steps=int(steps),
            guidance_scale=guidance,
            generator=gen
        ).frames[0]
        path = f"t2v_{uuid.uuid4().hex[:8]}.mp4"
        export_to_video(output, path, fps=16)
        clear_memory()
        return path, f"✅ Done! Saved as {path}"
    except Exception as e:
        return None, f"❌ Error: {e}"

def generate_i2v(image, prompt, steps, guidance, num_frames, seed, progress=gr.Progress(track_tqdm=True)):
    if image is None:
        return None, "❌ Please upload an image."
    try:
        pipe = load_pipeline("i2v")
        gen = torch.Generator("cuda" if torch.cuda.is_available() else "cpu")
        if seed != -1: gen.manual_seed(int(seed))
        output = pipe(
            image=image,
            prompt=prompt or "",
            num_frames=int(num_frames),
            num_inference_steps=int(steps),
            guidance_scale=guidance,
            generator=gen
        ).frames[0]
        path = f"i2v_{uuid.uuid4().hex[:8]}.mp4"
        export_to_video(output, path, fps=16)
        clear_memory()
        return path, f"✅ Done! Saved as {path}"
    except Exception as e:
        return None, f"❌ Error: {e}"

CSS = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.generate-btn { background: linear-gradient(90deg, #6366f1, #8b5cf6) !important; color: white !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="2M Cloud Video Generator") as demo:
    gr.Markdown("# 🎬 2M Cloud Video Generator\n**Wan 2.1 · device_map=auto · Colab & Kaggle T4**")

    with gr.Tab("📝 Text to Video"):
        with gr.Row():
            with gr.Column(scale=1):
                t2v_prompt    = gr.Textbox(label="Prompt", lines=3, placeholder="A majestic dragon soaring over snow-capped mountains, cinematic 4k...")
                t2v_neg       = gr.Textbox(label="Negative Prompt", value="low quality, blurry, static, watermark, text")
                with gr.Row():
                    t2v_steps     = gr.Slider(10, 50, value=25, step=1, label="Steps")
                    t2v_guidance  = gr.Slider(1.0, 15.0, value=5.0, step=0.5, label="Guidance Scale")
                with gr.Row():
                    t2v_frames    = gr.Slider(17, 97, value=49, step=8, label="Frames (≈ 3s–6s)")
                    t2v_seed      = gr.Number(label="Seed (-1 = random)", value=-1)
                t2v_btn = gr.Button("🎬 Generate Video", variant="primary", elem_classes="generate-btn")
                t2v_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column(scale=1):
                t2v_output = gr.Video(label="Output Video")
        t2v_btn.click(generate_t2v, [t2v_prompt, t2v_neg, t2v_steps, t2v_guidance, t2v_frames, t2v_seed], [t2v_output, t2v_status])

    with gr.Tab("🖼️ Image to Video"):
        with gr.Row():
            with gr.Column(scale=1):
                i2v_input     = gr.Image(type="pil", label="Input Image")
                i2v_prompt    = gr.Textbox(label="Motion Prompt (Optional)", placeholder="The person waves slowly and smiles...")
                with gr.Row():
                    i2v_steps     = gr.Slider(10, 50, value=25, step=1, label="Steps")
                    i2v_guidance  = gr.Slider(1.0, 15.0, value=5.0, step=0.5, label="Guidance Scale")
                with gr.Row():
                    i2v_frames    = gr.Slider(17, 97, value=49, step=8, label="Frames")
                    i2v_seed      = gr.Number(label="Seed (-1 = random)", value=-1)
                i2v_btn = gr.Button("🎬 Animate Image", variant="primary", elem_classes="generate-btn")
                i2v_status = gr.Textbox(label="Status", interactive=False)
            with gr.Column(scale=1):
                i2v_output = gr.Video(label="Output Video")
        i2v_btn.click(generate_i2v, [i2v_input, i2v_prompt, i2v_steps, i2v_guidance, i2v_frames, i2v_seed], [i2v_output, i2v_status])

demo.launch(share=True, debug=False)

## 📥 Step 4: Download Latest Video
Finds the most recently generated `.mp4` and offers an in-notebook preview + a download link.

In [ ]:
import glob, os
from IPython.display import HTML, display, FileLink
from base64 import b64encode

mp4s = sorted(glob.glob("*.mp4"), key=os.path.getmtime, reverse=True)
if mp4s:
    latest = mp4s[0]
    print(f"▶  Playing: {latest}")
    data = b64encode(open(latest, 'rb').read()).decode()
    display(HTML(f'<video width=640 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
    display(FileLink(latest, result_html_prefix="📥 Download: "))
else:
    print("No videos found yet — generate one from the UI above!")

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*